<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0;">Lab 03: Add Tool Nodes to Your LangGraph Agent</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">Create a tool-calling graph with conditional routing for the Contoso Travel data lookup</p>
</div>

**What We Learn:** How to add tool-calling capabilities using <code>@tool</code> decorators, conditional edges for routing between LLM and tool nodes, creating a complete data-driven travel assistant graph.

---


## 🧳 The Contoso Travel Story

Same data access problem — the concierge needs real inventory. With LangGraph, tools become nodes in the graph, and conditional edges decide when to call them.

- **The Problem:** The graph-based concierge can chat but can't access real data. The team needs to add tool-calling as explicit graph nodes with conditional routing.
- **This Lab Solves:** Defining tools with `@tool`, binding them to the LLM, adding a tool execution node, and using conditional edges to create the classic ReAct loop: llm → tools → llm → ... → END.


## 1. Setup

Load environment, create clients, initialize the LLM, and load the Contoso Travel CSV data.

---


In [ ]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.ai.projects import AIProjectClient
from langchain_openai import AzureChatOpenAI

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))
endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4.1-mini")

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)

# Initialize LLM
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")
base_endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"].split("/api/projects")[0]

llm = AzureChatOpenAI(
    azure_deployment=model_name,
    azure_endpoint=base_endpoint,
    azure_ad_token_provider=token_provider,
    api_version="2025-01-01",
)

# Load Contoso Travel data
data_path = "../../data/contoso-travel"
flights_df = pd.read_csv(f"{data_path}/flights.csv")
hotels_df = pd.read_csv(f"{data_path}/hotels.csv")
cars_df = pd.read_csv(f"{data_path}/car_rentals.csv")

print(f"✅ Connected to Foundry project")
print(f"   Model: {model_name}")
print(f"   Data: {len(flights_df)} flights, {len(hotels_df)} hotels, {len(cars_df)} car rentals")

## 2. Define Tools with `@tool`

LangChain's <code>@tool</code> decorator converts Python functions into tool definitions the LLM can call. Each tool searches our CSV data.

---


In [ ]:
from langchain_core.tools import tool


@tool
def search_flights(origin: str = "", destination: str = "", max_price: float = 10000) -> str:
    """Search Contoso Travel flight inventory by origin, destination, and price."""
    results = flights_df.copy()
    if origin:
        results = results[results["origin"].str.lower().str.contains(origin.lower())]
    if destination:
        results = results[results["destination"].str.lower().str.contains(destination.lower())]
    results = results[results["price_usd"] <= max_price]
    if results.empty:
        return json.dumps({"message": "No flights found matching your criteria."})
    return results.head(5).to_json(orient="records", indent=2)


@tool
def search_hotels(city: str = "", min_stars: int = 1, max_price: float = 10000) -> str:
    """Search Contoso Travel hotel inventory by city, star rating, and price."""
    results = hotels_df.copy()
    if city:
        results = results[results["city"].str.lower().str.contains(city.lower())]
    results = results[results["star_rating"] >= min_stars]
    results = results[results["price_per_night_usd"] <= max_price]
    if results.empty:
        return json.dumps({"message": "No hotels found matching your criteria."})
    return results.head(5).to_json(orient="records", indent=2)


@tool
def search_car_rentals(city: str = "", car_type: str = "") -> str:
    """Search Contoso Travel car rental inventory by city and car type."""
    results = cars_df.copy()
    if city:
        results = results[results["city"].str.lower().str.contains(city.lower())]
    if car_type:
        results = results[results["car_type"].str.lower().str.contains(car_type.lower())]
    if results.empty:
        return json.dumps({"message": "No car rentals found matching your criteria."})
    return results.head(5).to_json(orient="records", indent=2)


print("✅ Tools defined: search_flights, search_hotels, search_car_rentals")

## 3. Bind Tools to LLM

Binding tools tells the LLM what functions are available. The LLM can then generate tool calls in its responses.

---


In [ ]:
tools = [search_flights, search_hotels, search_car_rentals]
llm_with_tools = llm.bind_tools(tools)
print(f"✅ Bound {len(tools)} tools to LLM")

## 4. Define the Tool Node and Routing

The <code>ToolNode</code> automatically dispatches tool calls to the right function. The <code>should_continue</code> function routes the graph: if the LLM made tool calls, go to the tools node; otherwise, end.

---


In [ ]:
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import SystemMessage

# Tool execution node — automatically dispatches to the right tool
tool_node = ToolNode(tools)

CONCIERGE_INSTRUCTIONS = """You are the Contoso Travel concierge agent. You help customers plan trips by
searching real flight, hotel, and car rental inventory.

ALWAYS use the available tools to look up real data before responding.
When a customer asks about travel options, search the inventory first.
Present results clearly with prices, dates, and key details.
Be friendly, knowledgeable, and proactive."""


def llm_call(state: MessagesState):
    """Process conversation through the LLM with tool-calling enabled."""
    messages = [SystemMessage(content=CONCIERGE_INSTRUCTIONS)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


def should_continue(state: MessagesState):
    """Route to tools if the LLM made tool calls, otherwise end."""
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END


print("✅ Tool node and routing logic defined")

## 5. Build the Complete Graph

Wire everything together: LLM node, tool node, conditional routing, and the loop back from tools to LLM.

---


In [ ]:
graph = StateGraph(MessagesState)
graph.add_node("llm_call", llm_call)      # LLM with tool-calling
graph.add_node("tools", tool_node)          # Tool execution

graph.add_edge(START, "llm_call")
graph.add_conditional_edges("llm_call", should_continue, {"tools": "tools", END: END})
graph.add_edge("tools", "llm_call")         # Loop back after tool execution

agent = graph.compile()
print("✅ Tool-calling graph: START → llm → (tools → llm)* → END")

## 6. Test Flight Search

Ask the agent to find flights — it should invoke the <code>search_flights</code> tool and present real results.

---


In [ ]:
from langchain_core.messages import HumanMessage

result = agent.invoke(
    {"messages": [HumanMessage(content="Find me flights from Seattle to Paris under $900.")]}
)

print("✈️  Flight Search Results:")
print("-" * 50)
print(result["messages"][-1].content)

## 7. Test Multi-Tool Query

Ask a complex question that requires searching flights, hotels, and car rentals — the agent should call multiple tools.

---


In [ ]:
result = agent.invoke(
    {"messages": [HumanMessage(content="I need a complete trip to Paris — flights from Seattle, a 4-star hotel, and an SUV rental. What are my options?")]}
)

print("🧳 Multi-Tool Response:")
print("-" * 50)
print(result["messages"][-1].content)

## 8. Visualize the Graph

With tool nodes added, the graph now shows the ReAct loop — LLM calls tools, tools feed back to LLM.

---


In [ ]:
# Display the tool-calling graph structure
agent.get_graph().print_ascii()

## 9. Compare Three Approaches

How tool-calling works across the three agent paradigms:

---

| Aspect | Prompted (FunctionTool) | MAF (local functions) | LangGraph (@tool + nodes) |
|--------|------------------------|----------------------|---------------------------|
| Tool definition | JSON schema + Python function | Methods on BaseAgent | <code>@tool</code> decorated functions |
| Tool execution | Agent loop calls your function | Framework dispatches | <code>ToolNode</code> dispatches |
| Routing | Foundry-managed | Your code decides | Conditional edges |
| Loop control | Automatic | Manual | Graph edges (tools → llm) |
| Visibility | Response events | Code inspection | Graph visualization |


## 10. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2e8b57; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
<b>✅ What We Accomplished</b><br/>Built a complete tool-calling LangGraph agent with three data tools, conditional routing, and the ReAct loop. The agent searches real flight, hotel, and car rental data.
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #1565c0; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
<b>💡 Key Takeaway</b><br/>In LangGraph, tools are <b>graph nodes</b> and routing is <b>conditional edges</b>. The <code>ToolNode</code> handles dispatch, and the graph structure makes the ReAct loop explicit: llm → tools → llm → ... → END.
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #ff9800; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
<b>➡️ Next:</b> You now have three agent paradigms in your toolkit — prompted, MAF, and LangGraph. Each has its strengths, and the observability patterns you've learned apply to all of them.
</div>


In [ ]:
# Cleanup: close clients
project_client.close()
credential.close()
print("✅ Clients closed.")